In [76]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [77]:
# Indlæs data
GDP = pd.read_csv("GDP.csv", skiprows=4)
ODR = pd.read_csv("age-dependency-ratio-old.csv")
TFP = pd.read_csv("total-factor-productivity.csv")
HC_DATA = pd.read_excel("pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']] # Relevante kolonner fra filen HC_DATA (pwt110)
GEO = pd.read_excel("geo_cepii.xls")

Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [78]:
# Relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Relevante geografiske variable
GEO = GEO[['country', 'landlocked', 'continent', 'lat', 'area']].copy()

# Beregner BNP per capita ud fra rgdpe og befolkning
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Udtrækker observationsåret 2002 og beholder de variable, der skal bruges som initiale forklarende variable i tværsnittet
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()

# Omdøber variablerne, så det tydeligt fremgår, at de er målt i 2002
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# Udtrækker TFP i slutåret 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()

# Omdøber TFP-variablen for 2020
tfp2020.columns = ['country','TFP2020']

# Merger 2002- og 2020-data, så hvert land får både initial og slut-TFP
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# Konstruerer long-difference outcome som ændringen i log(TFP) fra 2002 til 2020
Tvaersnit['GrowthTFP'] = np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])

# Udtrækker old dependency ratio i startåret 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()

# Omdøber variablerne, så de matcher merge-strukturen
odr2002.columns = ['country', 'ODR2002']

# Sammenfletter ODR med tværsnittet, så hvert land får sin initiale ODR-værdi
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')

# Merge med tværsnittet
Tvaersnit = pd.merge(Tvaersnit, GEO, on='country', how='left')

# Fjern evt. manglende observationer efter merge
Tvaersnit = Tvaersnit.dropna()

In [79]:
print(Tvaersnit.describe())

           HC2002     TFP2002     GDPpc2002     TFP2020   GrowthTFP  \
count  109.000000  109.000000    109.000000  109.000000  109.000000   
mean     2.478313    0.673247  18535.566806    0.632882   -0.020754   
std      0.667948    0.292500  17781.070928    0.221001    0.301025   
min      1.088122    0.167584    728.561351    0.200132   -1.073842   
25%      2.037780    0.406043   4542.128613    0.453812   -0.222283   
50%      2.538953    0.652943  11293.164098    0.620056   -0.057211   
75%      2.963812    0.882618  33735.040538    0.818584    0.174879   
max      3.583555    1.394810  84592.225755    1.236640    0.942964   

          ODR2002  landlocked         lat          area  
count  109.000000  109.000000  109.000000  1.090000e+02  
mean    12.407788    0.174312   18.715229  9.838796e+05  
std      7.601902    0.381130   28.129811  2.169853e+06  
min      1.583193    0.000000  -44.283330  3.160000e+02  
25%      6.008945    0.000000    1.283333  5.107600e+04  
50%      8.6

# HC + log(GDP) + geo variabler

In [80]:
# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): const + ODR + HC2002 + log_GDPpc2002
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + lat
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + log_area
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat', 'log_area']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + landlocked
X6 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat', 'log_area', 'landlocked']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)       (3)        (4)        (5)        (6)    
-------------------------------------------------------------------------------
const          0.1137**   0.1095    0.8981***  0.9085***  0.4020     0.6052*   
               (0.0558)   (0.1371)  (0.2780)   (0.2758)   (0.3416)   (0.3434)  
ODR2002        -0.0108*** -0.0110** -0.0064    -0.0080    -0.0099*   -0.0102*  
               (0.0031)   (0.0044)  (0.0044)   (0.0053)   (0.0055)   (0.0055)  
HC2002                    0.0025    0.1423*    0.1395*    0.1304*    0.1573**  
                          (0.0639)  (0.0766)   (0.0770)   (0.0763)   (0.0727)  
log_GDPpc2002                       -0.1290*** -0.1290*** -0.1110*** -0.1336***
                                    (0.0377)   (0.0376)   (0.0387)   (0.0368)  
lat                                            0.0009     0.0015     0.0015    
                                               (0.0011)   (0.0012)   (0.0012)  
log_area                               

# HC + geo variabler

In [81]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'lat',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5,],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)       (3)       (4)        (5)    
-------------------------------------------------------------------
const          0.1137**   0.1095    0.1199    -0.3802*   -0.3700*  
               (0.0558)   (0.1371)  (0.1356)  (0.2090)   (0.2155)  
ODR2002        -0.0108*** -0.0110** -0.0126** -0.0142*** -0.0145***
               (0.0031)   (0.0044)  (0.0054)  (0.0054)   (0.0054)  
HC2002                    0.0025    -0.0004   0.0128     0.0135    
                          (0.0639)  (0.0636)  (0.0632)   (0.0633)  
lat                                 0.0009    0.0016     0.0017    
                                    (0.0012)  (0.0012)   (0.0012)  
log_area                                      0.0391***  0.0388*** 
                                              (0.0142)   (0.0144)  
landlocked                                               -0.0280   
                                                         (0.0764)  
R-squared      0.0749     0.0749    0.0800    0

# Log(GDP) + geo variabler 

In [82]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + log_GDPpc2002
X2 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)       (5)    
--------------------------------------------------------------------
const          0.1137**   0.8307***  0.8439***  0.3215    0.4518    
               (0.0558)   (0.2779)   (0.2756)   (0.3371)  (0.3473)  
ODR2002        -0.0108*** -0.0011    -0.0030    -0.0053   -0.0048   
               (0.0031)   (0.0039)   (0.0046)   (0.0050)  (0.0050)  
log_GDPpc2002             -0.0907*** -0.0916*** -0.0753** -0.0860***
                          (0.0320)   (0.0318)   (0.0325)  (0.0324)  
lat                                  0.0010     0.0016    0.0017    
                                     (0.0012)   (0.0012)  (0.0012)  
log_area                                        0.0322**  0.0302**  
                                                (0.0144)  (0.0147)  
landlocked                                                -0.0841   
                                                          (0.0775)  
R-squared      0.0749     0.1449 

# Regioner + Geo variabler

In [83]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

north_america = ['Canada', 'United States']
oceania = ['Australia', 'Fiji', 'New Zealand']
middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)       (2)      (3)       (4)        (5)    
-----------------------------------------------------------------
const          0.1137**   0.1384   0.1207   -0.3190**  -0.2900** 
               (0.0558)   (0.0905) (0.0967) (0.1477)   (0.1412)  
ODR2002        -0.0108*** -0.0140* -0.0136  -0.0163**  -0.0173** 
               (0.0031)   (0.0081) (0.0083) (0.0080)   (0.0080)  
Europe                    0.0555   0.0100   0.0226     0.0419    
                          (0.1005) (0.1306) (0.1198)   (0.1199)  
Asia                      0.0922*  0.0810*  0.0161     0.0234    
                          (0.0482) (0.0488) (0.0501)   (0.0494)  
Africa                    0.0102   0.0256   -0.0473    -0.0424   
                          (0.0770) (0.0767) (0.0798)   (0.0818)  
LatinAmerica              -0.0798  -0.0648  -0.1060    -0.1122   
                          (0.0754) (0.0822) (0.0774)   (0.0783)  
NorthAmerica              -0.0034  -0.0427  -0.2380*** -0.2290***
         

In [85]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

north_america = ['Canada', 'United States']
oceania = ['Australia', 'Fiji', 'New Zealand']
middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)        (5)    
---------------------------------------------------------------------
const          0.1137**   0.9272***  0.9079***  0.3899     0.5426    
               (0.0558)   (0.3164)   (0.3377)   (0.3852)   (0.3801)  
ODR2002        -0.0108*** -0.0052    -0.0052    -0.0095    -0.0117   
               (0.0031)   (0.0077)   (0.0078)   (0.0085)   (0.0086)  
HC2002                    0.1085     0.1103     0.1197     0.1615*   
                          (0.0894)   (0.0902)   (0.0882)   (0.0827)  
log_GDPpc2002             -0.1404*** -0.1393*** -0.1210*** -0.1425***
                          (0.0424)   (0.0437)   (0.0459)   (0.0439)  
Europe                    0.1735*    0.1509     0.1340     0.1928    
                          (0.1020)   (0.1411)   (0.1272)   (0.1294)  
Asia                      0.1895***  0.1826***  0.1056     0.1307    
                          (0.0648)   (0.0708)   (0.0794)   (0.0798)  
Africa             